In [2]:
from functools import partial
from pathlib import Path

import astroplan as ap
import astropy.units as u

from astropaul.database import database_connection, html_path
import astropaul.targetlistcreator as tlc
import astropaul.lbt as lbt
import astropaul.html as html
import astropaul.phase as ph
import astropaul.priority as pr

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
date_ranges = [
    ("2026-09-10", "2026-09-16"),
    ("2026-09-29", "2026-10-08"),
    ("2026-11-03", "2026-11-07"),
    ("2026-11-24", "2026-11-29"),
    ("2026-12-06", "2026-12-10"),
    ("2027-01-06", "2027-01-14"),
]

min_altitude = 35 * u.deg

session_names = [f"LBT Observing {utc_start}" for utc_start, _ in date_ranges]
html_paths = [(html_path() / name) for name in session_names]
for path in html_paths:
    html.clear_directory(path)

science_target_types = {"QuadEB", "SextEB"}
calibration_target_types = {"RV Standard", "Telluric Standard"}

phase_event_defs = [
    ph.PhaseEventDef("T", partial(ph.calc_time_of_phase, phase=0.0)),
    ph.PhaseEventDef("B", partial(ph.calc_time_of_phase, phase=0.05)),
    ph.PhaseEventDef("R", partial(ph.calc_time_of_phase, phase=0.18)),
    ph.PhaseEventDef("B", partial(ph.calc_time_of_phase, phase=0.32)),
    ph.PhaseEventDef("T", partial(ph.calc_time_of_phase, phase=0.45)),
    ph.PhaseEventDef("B", partial(ph.calc_time_of_phase, phase=0.55)),
    ph.PhaseEventDef("R", partial(ph.calc_time_of_phase, phase=0.68)),
    ph.PhaseEventDef("B", partial(ph.calc_time_of_phase, phase=0.82)),
    ph.PhaseEventDef("T", partial(ph.calc_time_of_phase, phase=0.95)),
]

pepsi_args = {
    "fiber": "300",
    "cd_blue": 3,
    "cd_red": 6,
    "snr": 100,
    "binocular": True,
    "priority": "(see grid)",
}

readme_name = "UVa_Multistar"
readme_header = Path(f"{readme_name}.header").read_text()
readme_footer = Path(f"{readme_name}.footer").read_text()

science_target_steps = [
    partial(tlc.filter_targets, criteria=lambda df: df["List LBT 2026B"]),
    partial(tlc.filter_targets, criteria=lambda df: df["Num Ephemerides"] == 4),
    # partial(tlc.filter_targets, criteria=lambda df: df["Num PEPSI Observations"] > 0),
    partial(tlc.filter_targets, inverse=True, criteria=lambda df: df["Teff"].isna()),
    tlc.add_pepsi_goals,
    partial(lbt.add_pepsi_params, **dict({"num_exp": 2}, **pepsi_args)),
]

calibration_target_steps = [
    partial(tlc.filter_targets, criteria=lambda df: df["Target Type"].isin(calibration_target_types)),
    partial(tlc.add_columns_from_sql, table_name="RV Calibration Targets", primary_cols=["RV"]),
    partial(tlc.filter_targets, inverse=True, criteria=lambda df: df["Teff"].isna()),
    partial(lbt.add_pepsi_params, **dict({"num_exp": 1}, **pepsi_args)),
]

common_steps = [
    partial(tlc.add_observability, observability_threshold=(min_altitude, 80 * u.deg)),
    partial(tlc.filter_targets, criteria=lambda df: (df["Observable Any Night"])),
    tlc.add_tess_catalog_associations,
    partial(tlc.filter_targets, criteria=lambda df: df["PEPSI exp_time"] < 600),
]

all_targets = tlc.TargetList.load()
target_lists, sessions, csv_tables, readmes = [], [], [], []
for (utc_start, utc_end), name, path in zip(date_ranges, session_names, html_paths):
    session = tlc.ObservingSession(ap.Observer.at_site("LBT"))
    session.add_day_range(utc_start, utc_end)
    sessions.append(session)

    with database_connection() as conn:
        creator = tlc.TargetListCreator(connection=conn, phase_event_defs=phase_event_defs, observing_session=session)
        science_targets = creator.calculate(initial_list=all_targets, name="Science Targets", steps=science_target_steps + common_steps)
        # temporarily disable adding calibration targets - we have enough data for now
        # calibration_targets = creator.calculate(
        #     initial_list=all_targets, name="Calibration Targets", steps=calibration_target_steps + common_steps, verbose=False
        # )
        # calibration_targets.other_lists = {}
        # tl = tlc.TargetList.union([science_targets, calibration_targets], name=name)
        tl = science_targets
        # tl = creator.calculate(initial_list=tl, steps=[partial(lbt.assign_rv_standards, target_types=science_target_types)])
    tl.target_list["PEPSI notes"] = tl.target_list["Target Type"]
    csv_table, readme = lbt.write_lbt_readme_file(readme_header, readme_footer, tl.target_list, session)
    csv_tables.append(csv_table.to_csv(index=False))
    readmes.append(readme)
    target_lists.append(tl)
    print(tl.summarize())

Name: Science Targets
Criteria:
  Target list loaded from file: TargetList_2026-08-31_14h10m32s.tl (750 targets)
  lambda df: df["List LBT 2026B"] (22 targets)
  lambda df: df["Num Ephemerides"] == 4 (10 targets)
  Inverse of: lambda df: df["Teff"].isna() (10 targets)
  Observability calculated at LBT in 15.0 min intervals from 2026-09-10 to 2026-09-16
    AltitudeConstraint: {'min': np.float64(35.0), 'max': np.float64(80.0), 'boolean_constraint': True}
  lambda df: (df["Observable Any Night"]) (10 targets)
  lambda df: df["PEPSI exp_time"] < 600 (10 targets)
  
10 targets:
    10 QuadEB
Column Count (primary, secondary):
    Target: (3, 4)
    RV Calibration Targets: (1, 2)
    List: (0, 20)
    Count: (8, 0)
    TESS Data: (4, 0)
    Gaia Bailer Jones: (1, 2)
    PEPSI : (3, 6)
    Observable: (4, 21)
Associated tables:
      53 rows,   3 columns: Catalog Membership
    1188 rows,   2 columns: List Memberships
     894 rows,   7 columns: Ephemerides
     716 rows, 126 columns: TESS
 

In [6]:
illumination_categories = [
    ((0.0, 0.4), "Dark"),
    ((0.4, 0.7), "Gray"),
    ((0.7, 1.0), "Bright"),
]

distance_categories = {
    "Dark": [
        ((0, 180), 1),
    ],
    "Gray": [
        ((0, 5), 0.1),
        ((5, 15), 0.85),
        ((15, 180), 1),
    ],
    "Bright": [
        ((0, 15), 0.25),
        ((15, 30), 0.75),
        ((30, 180), 1),
    ],
}

altitude_categories = [
    ((-90, min_altitude.value), 0),
    ((min_altitude.value, 45), 0.95),
    ((45, 90), 1),
]

for (utc_start, _), tl, session, path, csv_table, readme in zip(
    date_ranges, target_lists, sessions, html_paths, csv_tables, readmes
):
    science_tl = tl.copy()
    science_tl.target_list = science_tl.target_list[science_tl.target_list["Target Type"].isin(["QuadEB", "SextEB"])]
    pl = pr.PriorityList(science_tl, session, interval=30 * u.min)
    pr.calculate_moon_priority(pl, illumination_categories=illumination_categories, dist_categories=distance_categories)
    pr.calculate_altitude_priority(pl, altitude_categories=altitude_categories)
    pr.calculate_pepsi_priority(pl, phase_defs=phase_event_defs)
    pr.calculate_overall_priority(pl)
    pr.aggregate_target_priorities(pl, skip_column_threshold=0.0)
    pl.categorize_priorities(bins=[0.00, 0.20, 0.40, 0.6, 1.00], labels=["", "*", "* *", "* * *"])
    (Path(utc_start) / f"{readme_name}.csv").write_text(csv_table)
    (Path(utc_start) / f"{readme_name}.readme").write_text(readme)
    other_files = {
        "LBT Readme": readme,
        "LBT CSV": csv_table,
    }
    html.render_observing_pages(tl, pl, other_files, path)
    print(f"Wrote results for {utc_start} to {path}")

 '2026-09-10T03:00:00.000000000' '2026-09-10T03:30:00.000000000'
 '2026-09-10T04:00:00.000000000' '2026-09-10T04:30:00.000000000'
 '2026-09-10T05:00:00.000000000' '2026-09-10T05:30:00.000000000'
 '2026-09-10T06:00:00.000000000' '2026-09-10T06:30:00.000000000'
 '2026-09-10T07:00:00.000000000' '2026-09-10T07:30:00.000000000'
 '2026-09-10T08:00:00.000000000' '2026-09-10T08:30:00.000000000'
 '2026-09-10T09:00:00.000000000' '2026-09-10T09:30:00.000000000'
 '2026-09-10T10:00:00.000000000' '2026-09-10T10:30:00.000000000'
 '2026-09-10T11:00:00.000000000' '2026-09-10T11:30:00.000000000'
 '2026-09-10T12:00:00.000000000' '2026-09-10T12:30:00.000000000'], obsgeoloc=[( -93088.9013759 , -5373782.71174482, 3428163.41468467),
 ( 611129.93505012, -5340911.86323782, 3426323.28392995),
 (1304988.91822556, -5216154.24117267, 3424507.18060194),
 (1976550.92622576, -5001656.17037299, 3422746.34815308),
 (2614262.43338634, -4701107.86673046, 3421071.07922931),
 (3207152.27669299, -4319679.95087938, 3419510.1

Wrote results for 2026-09-10 to /Users/paul/Dropbox/Astro/Observing Files/LBT Observing 2026-09-10


 '2026-09-29T03:00:00.000000000' '2026-09-29T03:30:00.000000000'
 '2026-09-29T04:00:00.000000000' '2026-09-29T04:30:00.000000000'
 '2026-09-29T05:00:00.000000000' '2026-09-29T05:30:00.000000000'
 '2026-09-29T06:00:00.000000000' '2026-09-29T06:30:00.000000000'
 '2026-09-29T07:00:00.000000000' '2026-09-29T07:30:00.000000000'
 '2026-09-29T08:00:00.000000000' '2026-09-29T08:30:00.000000000'
 '2026-09-29T09:00:00.000000000' '2026-09-29T09:30:00.000000000'
 '2026-09-29T10:00:00.000000000' '2026-09-29T10:30:00.000000000'
 '2026-09-29T11:00:00.000000000' '2026-09-29T11:30:00.000000000'
 '2026-09-29T12:00:00.000000000' '2026-09-29T12:30:00.000000000'], obsgeoloc=[(1637617.24402372, -5122059.10819312, 3423628.31344781),
 (2294002.59625162, -4864836.70967786, 3421903.92193453),
 (2911076.18779473, -4523917.94227253, 3420279.58723092),
 (3478221.90960368, -4105167.95914409, 3418783.25565676),
 (3985682.6096956 , -3615790.91856754, 3417440.6713744 ),
 (4424727.95480569, -3064206.04387573, 3416274.9

Wrote results for 2026-09-29 to /Users/paul/Dropbox/Astro/Observing Files/LBT Observing 2026-09-29


 '2026-11-03T02:00:00.000000000' '2026-11-03T02:30:00.000000000'
 '2026-11-03T03:00:00.000000000' '2026-11-03T03:30:00.000000000'
 '2026-11-03T04:00:00.000000000' '2026-11-03T04:30:00.000000000'
 '2026-11-03T05:00:00.000000000' '2026-11-03T05:30:00.000000000'
 '2026-11-03T06:00:00.000000000' '2026-11-03T06:30:00.000000000'
 '2026-11-03T07:00:00.000000000' '2026-11-03T07:30:00.000000000'
 '2026-11-03T08:00:00.000000000' '2026-11-03T08:30:00.000000000'
 '2026-11-03T09:00:00.000000000' '2026-11-03T09:30:00.000000000'
 '2026-11-03T10:00:00.000000000' '2026-11-03T10:30:00.000000000'
 '2026-11-03T11:00:00.000000000' '2026-11-03T11:30:00.000000000'
 '2026-11-03T12:00:00.000000000' '2026-11-03T12:30:00.000000000'
 '2026-11-03T13:00:00.000000000'], obsgeoloc=[( 3250691.5481725 , -4287146.05924376, 3419341.1207044 ),
 ( 3783925.89834801, -3825987.52603611, 3417928.43997891),
 ( 4252216.46278544, -3299005.11746903, 3416684.20591061),
 ( 4647506.78973758, -2715265.01749315, 3415629.82457267),
 ( 4

...

Wrote results for 2026-11-24 to /Users/paul/Dropbox/Astro/Observing Files/LBT Observing 2026-11-24


 '2026-12-06T02:00:00.000000000' '2026-12-06T02:30:00.000000000'
 '2026-12-06T03:00:00.000000000' '2026-12-06T03:30:00.000000000'
 '2026-12-06T04:00:00.000000000' '2026-12-06T04:30:00.000000000'
 '2026-12-06T05:00:00.000000000' '2026-12-06T05:30:00.000000000'
 '2026-12-06T06:00:00.000000000' '2026-12-06T06:30:00.000000000'
 '2026-12-06T07:00:00.000000000' '2026-12-06T07:30:00.000000000'
 '2026-12-06T08:00:00.000000000' '2026-12-06T08:30:00.000000000'
 '2026-12-06T09:00:00.000000000' '2026-12-06T09:30:00.000000000'
 '2026-12-06T10:00:00.000000000' '2026-12-06T10:30:00.000000000'
 '2026-12-06T11:00:00.000000000' '2026-12-06T11:30:00.000000000'
 '2026-12-06T12:00:00.000000000' '2026-12-06T12:30:00.000000000'
 '2026-12-06T13:00:00.000000000' '2026-12-06T13:30:00.000000000'], obsgeoloc=[( 5047395.77847176, -1871763.7936386 , 3414499.48465048),
 ( 5249045.74674509, -1196230.71964404, 3413950.9938408 ),
 ( 5360546.64886644,  -500116.22395028, 3413639.40132197),
 ( 5379980.2281935 ,   204603.7

Wrote results for 2026-12-06 to /Users/paul/Dropbox/Astro/Observing Files/LBT Observing 2026-12-06


 '2027-01-06T02:00:00.000000000' '2027-01-06T02:30:00.000000000'
 '2027-01-06T03:00:00.000000000' '2027-01-06T03:30:00.000000000'
 '2027-01-06T04:00:00.000000000' '2027-01-06T04:30:00.000000000'
 '2027-01-06T05:00:00.000000000' '2027-01-06T05:30:00.000000000'
 '2027-01-06T06:00:00.000000000' '2027-01-06T06:30:00.000000000'
 '2027-01-06T07:00:00.000000000' '2027-01-06T07:30:00.000000000'
 '2027-01-06T08:00:00.000000000' '2027-01-06T08:30:00.000000000'
 '2027-01-06T09:00:00.000000000' '2027-01-06T09:30:00.000000000'
 '2027-01-06T10:00:00.000000000' '2027-01-06T10:30:00.000000000'
 '2027-01-06T11:00:00.000000000' '2027-01-06T11:30:00.000000000'
 '2027-01-06T12:00:00.000000000' '2027-01-06T12:30:00.000000000'
 '2027-01-06T13:00:00.000000000' '2027-01-06T13:30:00.000000000'], obsgeoloc=[( 5299415.54579504,   949394.90543077, 3413701.0356675 ),
 ( 5129661.19960468,  1633639.82376491, 3414133.48858374),
 ( 4871812.3932037 ,  2289781.10107829, 3414799.714326  ),
 ( 4530305.14789483,  2906530.5

Wrote results for 2027-01-06 to /Users/paul/Dropbox/Astro/Observing Files/LBT Observing 2027-01-06
